In [ ]:
import html
import re

import pandas as pd
from bs4 import BeautifulSoup

from sephora_test.logger import setup_logger

logger = setup_logger(__name__)

PATH_TRAINING_DATA = "/home/ubuntu/Desktop/sephora_test/data/training_set.csv"
training_data = pd.read_csv(PATH_TRAINING_DATA)
training_data.head()

In [ ]:
training_data["nature_code"].unique()

In [ ]:
from sephora_test.apply_transformation.clean_text import clean_text

training_data["cleanDescription"] = training_data["longDescription"].apply(
    clean_text
)
training_data["cleanName"] = training_data["productName"].apply(clean_text)

training_data = training_data[
    [
        "productId",
        "productName",
        "cleanName",
        "longDescription",
        "cleanDescription",
        "nature_code",
        "target_code",
    ]
]
training_data.head()

In [ ]:
logger.info(f"Training data shape: {training_data.shape}")
logger.info(f"Training data columns: {training_data.columns.tolist()}")

In [ ]:
def clean_description(text):
    if pd.isna(text):
        return ""

    # Decode HTML entities: &eacute; -> é, &nbsp; -> space, etc.
    text = html.unescape(text)

    # Remove HTML tags but keep their textual content
    text = BeautifulSoup(text, "lxml").get_text(" ", strip=True)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
training_data["cleanDescription"] = training_data["longDescription"].apply(
    clean_description
)
before_cleaning = (
    training_data[["longDescription", "target_code"]].iloc[0].to_dict()
)
after_cleaning = (
    training_data[["cleanDescription", "target_code"]].iloc[0].to_dict()
)
before_cleaning

In [ ]:
training_data.head()

In [ ]:
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download stopwords and tokenizer if you haven't already
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

In [ ]:
type(string.punctuation)

In [ ]:
penctuations = set(string.punctuation + "«»“”‘’—…")
frensh_stopwords = set(stopwords.words("french"))

In [ ]:
def get_most_common_words(
    nature_code: str = "nature_code",
    clean_description: str = "cleanDescription",
    top_n: int = 10,
):
    """
    Get the most common words for each nature_code in the training data.

    Args:
        nature_code (str): The column name for nature_code.
        clean_description (str): The column name for cleanDescription.
        top_n (int): The number of top common words to return.

    Returns:
        dict: A dictionary where keys are nature_codes and values are lists of tuples (word, count).
    """
    common_words = {}

    for code in training_data[nature_code].unique():
        # Filter the data for the current nature_code
        filtered_data = training_data[training_data[nature_code] == code]

        # Join all clean descriptions into a single string
        all_text = " ".join(filtered_data[clean_description].dropna())
        for p in penctuations:
            all_text = all_text.replace(p, "")
        all_text = all_text.lower()
        all_text = word_tokenize(all_text)
        all_text = [word for word in all_text if word not in frensh_stopwords]

        # Split into words and count occurrences
        word_counts = Counter(all_text)

        # Get the most common words
        common_words[code] = word_counts.most_common(top_n)

    return common_words

In [ ]:
print(string.punctuation)

In [ ]:
common_words_by_nature_code = get_most_common_words(
    nature_code="target_code", clean_description="cleanDescription", top_n=20
)
from pprint import pprint

pprint(common_words_by_nature_code)

In [ ]:
import pandas as pd

from sephora_test.apply_transformation.label_construction import (
    LabelConstructor,
)
from sephora_test.config.labels_map import (
    mapping_nature_code,
    mapping_target_code,
)

In [ ]:
nature_constructor = LabelConstructor(
    label_map=mapping_nature_code,
    columns_list_to_use=[
        "cleanName",
        "cleanDescription",
    ],
    output_column_name="nature_code_predicted",
)

df = nature_constructor.NatureConstructor(training_data)
df.head()

In [ ]:
target_constructor = LabelConstructor(
    label_map=mapping_target_code,
    columns_list_to_use=[
        "cleanName",
        "cleanDescription",
    ],
    output_column_name="target_code_predicted",
)

df = target_constructor.TargetConstructor(df)
print(df.head())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)

eval_df = df[["nature_code", "nature_code_predicted"]].dropna()

y_true_nature = eval_df["nature_code"]
y_pred_nature = eval_df["nature_code_predicted"]

nature_accuracy = accuracy_score(
    y_true_nature,
    y_pred_nature,
)

nature_f1_macro = f1_score(
    y_true_nature,
    y_pred_nature,
    average="macro",
)

nature_f1_weighted = f1_score(
    y_true_nature,
    y_pred_nature,
    average="weighted",
)

print("Nature code evaluation")
print("----------------------")
print(f"Accuracy: {nature_accuracy:.4f}")
print(f"Macro F1: {nature_f1_macro:.4f}")
print(f"Weighted F1: {nature_f1_weighted:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true_nature,
        y_pred_nature,
        zero_division=0,
    )
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)

eval_df_target = df[["target_code", "target_code_predicted"]].dropna()

y_true_target = eval_df_target["target_code"]
y_pred_target = eval_df_target["target_code_predicted"]

target_accuracy = accuracy_score(
    y_true_target,
    y_pred_target,
)

target_f1_macro = f1_score(
    y_true_target,
    y_pred_target,
    average="macro",
)

target_f1_weighted = f1_score(
    y_true_target,
    y_pred_target,
    average="weighted",
)

print("Target code evaluation")
print("----------------------")
print(f"Accuracy: {target_accuracy:.4f}")
print(f"Macro F1: {target_f1_macro:.4f}")
print(f"Weighted F1: {target_f1_weighted:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true_target,
        y_pred_target,
        zero_division=0,
    )
)